In [ ]:
#konfiguracija
import os, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

#kad rezultatai butu vienodi po kiekvieno paleidimo
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass
    print(f"TensorFlow: {tf.__version__}  (seed={SEED})")
except ImportError:
    raise RuntimeError("Reikalinga tensorflow: pip install tensorflow")

#datos
TRAIN_START  = pd.Timestamp("2020-04-01")
EFF_START    = pd.Timestamp("2021-03-31")   #po t-364 NaN
BLIND_START  = pd.Timestamp("2026-01-01")
BLIND_END    = pd.Timestamp("2026-03-31")

#failai
CSV_PATH     = "duomenys.csv"
LOG_PATH     = "GRU_Zurnalas.csv"
PARAMS_PATH  = "GRU_best_params.json"
SCALER_PATH  = "GRU_scaler.json"

#paskutine diena su faktais
_raw_csv = pd.read_csv(CSV_PATH)
_raw_csv.columns = ['Data'] + list(_raw_csv.columns[1:])
_raw_csv['Data'] = pd.to_datetime(_raw_csv['Data'], dayfirst=True)
_col_b = _raw_csv.iloc[:, 1]
_mask = _col_b.notna() & (_col_b != '') & (_col_b != 0)
TRAIN_END = pd.Timestamp(_raw_csv.loc[_mask, 'Data'].max())
del _raw_csv, _col_b, _mask
print(f'Mokymo periodas: {TRAIN_START.date()} -> {TRAIN_END.date()}')

#lag zingsniai (tokie patys kaip xgboost)
LAGS = (1, 2, 3, 7, 14, 28, 364)

#mc dropout prognoziu intervalams
MC_DROPOUT_SAMPLES = 100
CI_ALPHA = 0.10   #90% CI

#metrikos
def mae(y, yhat):  return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))
def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat))**2)))
def mape(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    mask = y != 0
    return float(np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100) if mask.any() else np.nan
def smape(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    denom = (np.abs(y) + np.abs(yhat)) / 2.0
    mask = denom != 0
    return float(np.mean(np.abs(y[mask] - yhat[mask]) / denom[mask]) * 100) if mask.any() else np.nan
def mase(y, yhat, y_hist, m=7):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    y_hist = np.asarray(y_hist, dtype=float)
    denom = float(np.mean(np.abs(y_hist[m:] - y_hist[:-m])))
    return float(np.mean(np.abs(y - yhat)) / denom) if denom > 0 else np.nan

def savaites_nr_metuose(data: pd.Timestamp) -> int:
    jan1 = pd.Timestamp(year=data.year, month=1, day=1)
    pirmas_sekm = jan1 + pd.Timedelta(days=(6 - jan1.dayofweek) % 7)
    if data <= pirmas_sekm:
        return 1
    return 2 + (data - (pirmas_sekm + pd.Timedelta(days=1))).days // 7

print(f"Konfig: mokom {TRAIN_START.date()} -> {TRAIN_END.date()}, blind {BLIND_START.date()} -> {BLIND_END.date()}")
print(f"Lag zingsniai: {LAGS}")

#protingas keso nuvertinimas - kad nebrangus hyperband nebebutu paleidziamas veltui
_needs_refresh = False
if os.path.exists(PARAMS_PATH):
    try:
        with open(PARAMS_PATH, "r", encoding="utf-8") as _f:
            _saved_end = json.load(_f).get("_train_end", None)
        if _saved_end != str(TRAIN_END.date()):
            _needs_refresh = True
            print(f"Duomenys pasikeite ({_saved_end} -> {TRAIN_END.date()}); JSON atnaujinami.")
    except Exception:
        _needs_refresh = True
elif os.path.exists(SCALER_PATH):
    _needs_refresh = True

if _needs_refresh:
    for _cf in (PARAMS_PATH, SCALER_PATH):
        if os.path.exists(_cf):
            os.remove(_cf)
            print(f"Istrintas: {_cf}")
elif os.path.exists(PARAMS_PATH) and os.path.exists(SCALER_PATH):
    print(f"JSON failai atitinka TRAIN_END={TRAIN_END.date()}; bus pakartotinai naudojami.")
else:
    print(f"JSON failu dar nera; bus apmokoma is naujo.")


In [ ]:
#duomenys ir pozymiai (toks pat rinkinys kaip xgboost - kad palyginimas butu sazingas)

#nuskaitymas
df_raw = pd.read_csv(CSV_PATH)
df_raw["Data"] = pd.to_datetime(df_raw["Data"], dayfirst=True, errors="coerce")
df_raw = df_raw.dropna(subset=["Data"]).sort_values("Data").set_index("Data")

paskutine_CSV = df_raw.index.max()
pilnas_indeksas = pd.date_range(TRAIN_START, paskutine_CSV, freq="D")
df = df_raw.reindex(pilnas_indeksas)
df.index.name = "Data"
print(f"Duomenys: {TRAIN_START.date()} -> {paskutine_CSV.date()} ({len(df)} d., truksta: {int(df['Skambuciai'].isna().sum())})")

#trukumu uzpildymas
df["Skambuciai"] = df["Skambuciai"].interpolate(method="linear", limit=3, limit_area="inside")
for dt in df.index[df["Skambuciai"].isna()]:
    kand = []
    for y in range(TRAIN_START.year, dt.year):
        past_date = dt - pd.DateOffset(years=(dt.year - y))
        if past_date in df.index and not pd.isna(df.at[past_date, "Skambuciai"]):
            kand.append(float(df.at[past_date, "Skambuciai"]))
    if kand:
        df.at[dt, "Skambuciai"] = float(np.median(kand))
df["Skambuciai"] = df["Skambuciai"].fillna(df["Skambuciai"].shift(1).rolling(7, min_periods=1).mean())

#anomalijos tik mokymo lange
tm = df.loc[(df.index >= TRAIN_START) & (df.index <= TRAIN_END), "Skambuciai"]
sv = tm.shift(1).rolling(30, min_periods=7).mean()
ss = tm.shift(1).rolling(30, min_periods=7).std()
anomalijos = (((tm - sv).abs() > 3 * ss) & ss.notna()).sum()
print(f"Anomaliju kandidatai (>3 sigma): {int(anomalijos)}")

#LT sventes
LT_SVENTES = pd.to_datetime([
    "2020-01-01","2021-01-01","2022-01-01","2023-01-01","2024-01-01","2025-01-01","2026-01-01",
    "2020-02-16","2021-02-16","2022-02-16","2023-02-16","2024-02-16","2025-02-16","2026-02-16",
    "2020-03-11","2021-03-11","2022-03-11","2023-03-11","2024-03-11","2025-03-11","2026-03-11",
    "2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05",
    "2020-04-13","2021-04-05","2022-04-18","2023-04-10","2024-04-01","2025-04-21","2026-04-06",
    "2020-05-01","2021-05-01","2022-05-01","2023-05-01","2024-05-01","2025-05-01","2026-05-01",
    "2020-06-24","2021-06-24","2022-06-24","2023-06-24","2024-06-24","2025-06-24","2026-06-24",
    "2020-07-06","2021-07-06","2022-07-06","2023-07-06","2024-07-06","2025-07-06","2026-07-06",
    "2020-08-15","2021-08-15","2022-08-15","2023-08-15","2024-08-15","2025-08-15","2026-08-15",
    "2020-11-01","2021-11-01","2022-11-01","2023-11-01","2024-11-01","2025-11-01","2026-11-01",
    "2020-11-02","2021-11-02","2022-11-02","2023-11-02","2024-11-02","2025-11-02","2026-11-02",
    "2020-12-24","2021-12-24","2022-12-24","2023-12-24","2024-12-24","2025-12-24","2026-12-24",
    "2020-12-25","2021-12-25","2022-12-25","2023-12-25","2024-12-25","2025-12-25","2026-12-25",
    "2020-12-26","2021-12-26","2022-12-26","2023-12-26","2024-12-26","2025-12-26","2026-12-26",
])
VELYKOS  = pd.to_datetime(["2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05"])
SEKMINES = pd.to_datetime(["2020-05-31","2021-05-23","2022-06-05","2023-05-28","2024-05-19","2025-06-08","2026-05-24"])
SVENT_SET = set(LT_SVENTES); VEL_SET = set(VELYKOS); SEK_SET = set(SEKMINES)

#pozymiai (identiska xgboost - kad palyginimas butu sazingas)
def sukurti_pozymius(series: pd.DataFrame) -> pd.DataFrame:
    d = series.copy()
    idx = pd.Series(d.index, index=d.index)
    dow = d.index.dayofweek
    mon = d.index.month
    d["week_iso"]  = d.index.isocalendar().week.astype(int)
    d["day"]       = d.index.day
    d["quarter"]   = d.index.quarter
    d["dayofyear"] = d.index.dayofyear
    d["is_weekend"] = (dow >= 5).astype(int)
    d["is_holiday"]         = idx.isin(SVENT_SET).astype(int).values
    d["day_before_holiday"] = (idx + pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
    d["day_after_holiday"]  = (idx - pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
    d["is_easter"]          = idx.isin(VEL_SET).astype(int).values
    d["is_pentecost"]       = idx.isin(SEK_SET).astype(int).values
    d["dow_sin"] = np.sin(2*np.pi*dow/7); d["dow_cos"] = np.cos(2*np.pi*dow/7)
    d["moy_sin"] = np.sin(2*np.pi*mon/12); d["moy_cos"] = np.cos(2*np.pi*mon/12)
    for lag in LAGS:
        d[f"lag_{lag}"] = d["Skambuciai"].shift(lag)
    s1 = d["Skambuciai"].shift(1)
    d["roll_mean_7"]  = s1.rolling(7).mean()
    d["roll_mean_14"] = s1.rolling(14).mean()
    d["roll_mean_30"] = s1.rolling(30).mean()
    d["roll_std_7"]   = s1.rolling(7).std()
    d["roll_med_7"]   = s1.rolling(7).median()
    d["roll_min_7"]   = s1.rolling(7).min()
    d["roll_max_7"]   = s1.rolling(7).max()
    return d

df_feat = sukurti_pozymius(df[["Skambuciai"]])
FEATURE_COLS = [c for c in df_feat.columns if c != "Skambuciai"]
print(f"Pozymiu skaicius: {len(FEATURE_COLS)}")
print(f"Pirma diena su visais lag: {df_feat.dropna(subset=[f'lag_{l}' for l in LAGS]).index.min().date()}")


In [ ]:
#normalizavimas, sekos, baziniai modeliai

#efektyvus mokymo rinkinys
D_eff = df_feat[(df_feat.index >= EFF_START) & (df_feat.index <= TRAIN_END)] \
           .dropna(subset=[f"lag_{l}" for l in LAGS])
print(f"D_eff: {D_eff.index.min().date()} -> {D_eff.index.max().date()} ({len(D_eff)} d.)")

#min-max scaler - tik viena karta, fit ant mokymo (kad nebutu data leakage)
if os.path.exists(SCALER_PATH):
    with open(SCALER_PATH, "r", encoding="utf-8") as f:
        SCALER = json.load(f)
    print(f"Scaler pakrautas is {SCALER_PATH}")
else:
    feat_min = D_eff[FEATURE_COLS].min().to_dict()
    feat_max = D_eff[FEATURE_COLS].max().to_dict()
    y_min = float(D_eff["Skambuciai"].min())
    y_max = float(D_eff["Skambuciai"].max())
    SCALER = {
        "_train_end": str(TRAIN_END.date()),
        "feat_min": feat_min,
        "feat_max": feat_max,
        "y_min":    y_min,
        "y_max":    y_max,
        "feature_cols": FEATURE_COLS,
    }
    with open(SCALER_PATH, "w", encoding="utf-8") as f:
        json.dump(SCALER, f, indent=2)
    print(f"Scaler issaugotas: {SCALER_PATH}")
print(f"   y skale: [{SCALER['y_min']:.0f}, {SCALER['y_max']:.0f}]")

def scale_features(X: np.ndarray) -> np.ndarray:
    lo = np.array([SCALER["feat_min"][c] for c in FEATURE_COLS], dtype=float)
    hi = np.array([SCALER["feat_max"][c] for c in FEATURE_COLS], dtype=float)
    rng = np.where(hi - lo == 0, 1.0, hi - lo)
    return (X - lo) / rng

def scale_y(y):
    return (np.asarray(y, dtype=float) - SCALER["y_min"]) / max(1e-9, SCALER["y_max"] - SCALER["y_min"])

def inverse_scale_y(y_s):
    return np.asarray(y_s, dtype=float) * (SCALER["y_max"] - SCALER["y_min"]) + SCALER["y_min"]

#sekos - trimatis masyvas (n, lookback, feat_n)
def sukurti_sekas(feat_df: pd.DataFrame, lookback: int):
    X_raw = feat_df[FEATURE_COLS].values.astype(float)
    y_raw = feat_df["Skambuciai"].values.astype(float)
    X_norm = scale_features(X_raw)
    y_norm = scale_y(y_raw)
    n = len(feat_df)
    if n < lookback:
        return np.empty((0, lookback, len(FEATURE_COLS))), np.empty((0,)), feat_df.index[:0]
    X_seq = np.stack([X_norm[i - lookback + 1: i + 1] for i in range(lookback - 1, n)])
    y_seq = y_norm[lookback - 1: n]
    idx   = feat_df.index[lookback - 1: n]
    return X_seq, y_seq, idx

#baziniai modeliai 2026 metams
faktai_2026 = df_feat[(df_feat.index >= BLIND_START) &
                      (df_feat.index <= BLIND_END) &
                      df_feat["Skambuciai"].notna()]

if len(faktai_2026):
    y_true = faktai_2026["Skambuciai"].astype(float).values
    hist   = D_eff["Skambuciai"].astype(float).values
    baziniai = {
        "Naive (t-1)":            faktai_2026["lag_1"].values,
        "Seasonal naive (t-7)":   faktai_2026["lag_7"].values,
        "7d slenk. vidurkis":     faktai_2026["roll_mean_7"].values,
        "Same week last year":    faktai_2026["lag_364"].values,
    }
    print(f"\n=== Baziniai modeliai 2026 m. ({len(faktai_2026)} d.) ===")
    print(f'{"Modelis":26s} {"MAE":>8s} {"RMSE":>8s} {"MAPE%":>8s} {"MASE":>8s}')
    for pav, yp in baziniai.items():
        yp_arr = np.asarray(yp, dtype=float)
        msk = ~np.isnan(yp_arr)
        if msk.any():
            print(f"{pav:26s} {mae(y_true[msk], yp_arr[msk]):8.1f} "
                  f"{rmse(y_true[msk], yp_arr[msk]):8.1f} "
                  f"{mape(y_true[msk], yp_arr[msk]):8.2f} "
                  f"{mase(y_true[msk], yp_arr[msk], hist):8.2f}")
else:
    print("\n2026 faktu dar nera.")


In [ ]:
#hiperparametru paieska su keras tuner hyperband
#brangus zingsnis - vyksta vieną karta, paskui paliekama atminty

import keras_tuner as kt
import os, json
from tensorflow.keras import layers, models, optimizers, losses, callbacks

#modelio kurimo funkcija (keras tuner ja kvies su skirtingais hp)
def build_model(hp):
    lb = hp.Int("lookback", 7, 28, step=7)

    model = models.Sequential()
    model.add(layers.Input(shape=(lb, len(FEATURE_COLS))))

    n_layers = hp.Int("n_layers", 1, 3)
    for i in range(n_layers):
        model.add(layers.GRU(
            units=hp.Choice("units", [32, 64, 128, 256]),
            return_sequences=(i < n_layers - 1),
            dropout=hp.Choice("dropout", [0.2, 0.3, 0.4]),
            recurrent_dropout=hp.Choice("rec_dropout", [0.0, 0.1])
        ))

    model.add(layers.Dense(1, activation="linear"))

    lr = hp.Choice("learning_rate", [0.001, 0.005, 0.01])
    loss_name = hp.Choice("loss", ["mse", "huber"])
    loss_fn = losses.Huber() if loss_name == "huber" else "mse"

    model.compile(optimizer=optimizers.Adam(learning_rate=lr, clipnorm=1.0), loss=loss_fn)
    return model


#custom tuner - leidzia keisti duomenu forma pagal lookback
class MyTuner(kt.Hyperband):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters

        model = self.hypermodel.build(hp)
        lb = hp.get("lookback")

        #sugeneruojam duomenis tikslia siai langu lygiai
        X_all, y_all, _ = sukurti_sekas(D_eff, lb)
        split = int(len(X_all) * 0.85)
        X_tr, y_tr = X_all[:split], y_all[:split]
        X_va, y_va = X_all[split:], y_all[split:]

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_va, y_va),
            batch_size=hp.Int("batch_size", 16, 64, step=16),
            epochs=25,
            verbose=0,
            callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
        )

        #grazinam mase original skaleje
        y_hat_s = model.predict(X_va, verbose=0).flatten()
        y_hat = inverse_scale_y(y_hat_s)
        y_va_orig = inverse_scale_y(y_va)
        y_hist_tr = inverse_scale_y(y_tr)

        val_mase = mase(y_va_orig, y_hat, y_hist_tr)

        return val_mase


#paleidimas
if os.path.exists(PARAMS_PATH):
    with open(PARAMS_PATH, "r", encoding="utf-8") as f:
        best_params = json.load(f)
    best_params.pop("_train_end", None)
    print(f"Hiperparametrai pakrauti is {PARAMS_PATH}")
else:
    print("Paleidziam keras tuner paieska...")
    tuner = MyTuner(
        hypermodel=build_model,
        objective=kt.Objective("val_mase", direction="min"),
        max_epochs=25,
        factor=3,
        directory="kt_dir",
        project_name="gru_search",
        overwrite=True,
        seed=SEED
    )

    tuner.search()

    best_hp = tuner.get_best_hyperparameters()[0]
    best_params = best_hp.values

    with open(PARAMS_PATH, "w", encoding="utf-8") as f:
        json.dump({"_train_end": str(TRAIN_END.date()), **best_params}, f, indent=2)
    print(f"\nPaieska baigta. Issaugot: {PARAMS_PATH}")

print("\nGeriausi parametrai:")
for k, v in best_params.items():
    print(f"  {k:14s} {v}")


In [ ]:
#mokymas + rekursyvi savaites prognoze + zurnalas

from tensorflow.keras import layers, models, optimizers, losses, callbacks
import pandas as pd
import numpy as np
import os

#galutinio modelio kurimas su fiksuotais hp
def build_gru(lookback: int, n_features: int, hp: dict) -> models.Model:
    inp = layers.Input(shape=(lookback, n_features))
    x = inp
    for i in range(hp["n_layers"]):
        ret_seq = (i < hp["n_layers"] - 1)
        x = layers.GRU(
            hp["units"],
            return_sequences=ret_seq,
            dropout=hp["dropout"],
            recurrent_dropout=hp["rec_dropout"],
            activation="tanh",
            recurrent_activation="sigmoid",
        )(x)
    out = layers.Dense(1, activation="linear")(x)
    m = models.Model(inp, out)
    loss_fn = losses.Huber(delta=1.0) if hp.get("loss", "mse") == "huber" else losses.MeanSquaredError()
    m.compile(
        optimizer=optimizers.Adam(learning_rate=hp["learning_rate"], clipnorm=1.0),
        loss=loss_fn,
    )
    return m

LOOKBACK = int(best_params["lookback"])

#mokymo rinkinys iki paskutines turimos dienos
mokymo = df_feat[(df_feat.index >= EFF_START) &
                 df_feat["Skambuciai"].notna()] \
             .dropna(subset=[f"lag_{l}" for l in LAGS])
paskutine_turima = mokymo.index.max()
print(f"Mokymo rinkinys: {mokymo.index.min().date()} -> {paskutine_turima.date()} ({len(mokymo)} d.)")

X_seq, y_seq, idx_seq = sukurti_sekas(mokymo, LOOKBACK)
print(f"Sekos: X={X_seq.shape}, y={y_seq.shape}")

#chronologinis 15% holdout - tik early stoppingui
split = int(len(X_seq) * 0.85)
X_tr_s, y_tr_s = X_seq[:split], y_seq[:split]
X_va_s, y_va_s = X_seq[split:], y_seq[split:]

#mokymas
tf.random.set_seed(SEED)
modelis = build_gru(LOOKBACK, len(FEATURE_COLS), best_params)
es = callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor="val_loss")
hist = modelis.fit(
    X_tr_s, y_tr_s,
    validation_data=(X_va_s, y_va_s),
    epochs=100,
    batch_size=int(best_params["batch_size"]),
    verbose=0,
    callbacks=[es],
)
print(f"GRU apmokytas: {len(hist.history['loss'])} epochos, val_loss={min(hist.history['val_loss']):.5f}")

def kita_savaite(paskutine: pd.Timestamp):
    start = paskutine + pd.Timedelta(days=1)
    if start > BLIND_END: return None
    days_until_sunday = 6 - start.dayofweek
    week_end = start + pd.Timedelta(days=days_until_sunday)
    week_end = min(week_end, BLIND_END)
    return pd.date_range(start, week_end, freq="D")

def mc_dropout_predict(model, x_seq: np.ndarray, n_samples: int = MC_DROPOUT_SAMPLES):
    #100 stochastiniu forward pass su iskirtu dropout
    preds = np.stack([
        model(x_seq, training=True).numpy().flatten()
        for _ in range(n_samples)
    ])
    mean_s = preds.mean(axis=0)
    lo_s   = np.quantile(preds, CI_ALPHA / 2, axis=0)
    hi_s   = np.quantile(preds, 1 - CI_ALPHA / 2, axis=0)
    return inverse_scale_y(mean_s), inverse_scale_y(lo_s), inverse_scale_y(hi_s)

def prognozuoti_savaite(model, istorija: pd.DataFrame, savaite):
    d = istorija[["Skambuciai"]].copy()
    trukstamos = [dt for dt in savaite if dt not in d.index]
    if trukstamos:
        d = pd.concat([d, pd.DataFrame({"Skambuciai": np.nan}, index=trukstamos)]).sort_index()
    prog = {}
    for dt in savaite:
        feat = sukurti_pozymius(d)
        window = feat.loc[:dt].tail(LOOKBACK)
        if len(window) < LOOKBACK:
            pred_y = float(d["Skambuciai"].dropna().tail(7).mean())
            prog[dt] = {"mean": pred_y, "lo": max(0.0, pred_y), "hi": max(0.0, pred_y)}
            d.at[dt, "Skambuciai"] = pred_y
            continue
        X_raw = window[FEATURE_COLS].fillna(0.0).values.astype(float)
        X_norm = scale_features(X_raw).reshape(1, LOOKBACK, len(FEATURE_COLS))
        mean_arr, lo_arr, hi_arr = mc_dropout_predict(model, X_norm)
        pred_y = float(mean_arr[0])
        prog[dt] = {"mean": pred_y, "lo": max(0.0, float(lo_arr[0])), "hi": float(hi_arr[0])}
        d.at[dt, "Skambuciai"] = pred_y
    return prog

#zurnalas
STULP = ["Savaites_Nr", "Prognoze", "CI_Lower", "CI_Upper", "Faktas", "Abs_Klaida", "Proc_Klaida"]
if os.path.exists(LOG_PATH):
    zurnalas = pd.read_csv(LOG_PATH, index_col="Data", parse_dates=True)
    for c in STULP:
        if c not in zurnalas.columns: zurnalas[c] = np.nan
    zurnalas = zurnalas[STULP]
else:
    zurnalas = pd.DataFrame(columns=STULP)
    zurnalas.index = pd.DatetimeIndex([], name="Data")
print(f"Zurnale: {len(zurnalas)} irasu")

#uzpildom faktais
uzpildyta = 0
for dt in zurnalas.index:
    if pd.isna(zurnalas.at[dt, "Faktas"]) and dt in df_feat.index \
            and not pd.isna(df_feat.at[dt, "Skambuciai"]):
        f = float(df_feat.at[dt, "Skambuciai"])
        p = float(zurnalas.at[dt, "Prognoze"])
        zurnalas.at[dt, "Faktas"]      = f
        zurnalas.at[dt, "Abs_Klaida"]  = abs(f - p)
        zurnalas.at[dt, "Proc_Klaida"] = abs(f - p) / f * 100 if f != 0 else np.nan
        uzpildyta += 1
if uzpildyta:
    print(f"Uzpildyta {uzpildyta} faktu")

#paskutines savaites paklaida
uzbaigta = zurnalas.dropna(subset=["Faktas"])
if len(uzbaigta):
    pask_nr = int(uzbaigta["Savaites_Nr"].max())
    grp = uzbaigta[uzbaigta["Savaites_Nr"] == pask_nr]
    if len(grp) >= 1:
        y_f = grp["Faktas"].values.astype(float)
        y_p = grp["Prognoze"].values.astype(float)
        print(f"\n--- Praejusi savaite {pask_nr}: {grp.index.min().date()} -> {grp.index.max().date()} ({len(grp)} d.) ---")
        print(f"  MAE  : {mae(y_f, y_p):8.1f}")
        print(f"  RMSE : {rmse(y_f, y_p):8.1f}")
        print(f"  MAPE : {mape(y_f, y_p):8.2f}%")
        print(f"  MASE : {mase(y_f, y_p, D_eff['Skambuciai'].values):8.2f}")

#nauja prognoze
savaite = kita_savaite(paskutine_turima)
if savaite is None:
    print("\n2026 prognozavimas baigtas (pasieke 2026-03-31).")
else:
    pirm, sekm = savaite[0], savaite[-1]
    sav_nr = savaites_nr_metuose(pirm)
    print(f"\nProgostozuojam savaite {sav_nr} ({pirm.year}): {pirm.date()} -> {sekm.date()} ({len(savaite)} d.)")
    prog = prognozuoti_savaite(modelis, df_feat, savaite)
    for dt in savaite:
        zurnalas.loc[dt] = {
            "Savaites_Nr": sav_nr,
            "Prognoze":    int(round(prog[dt]["mean"])),
            "CI_Lower":    int(round(prog[dt]["lo"])),
            "CI_Upper":    int(round(prog[dt]["hi"])),
            "Faktas":      np.nan,
            "Abs_Klaida":  np.nan,
            "Proc_Klaida": np.nan,
        }
    print("\nDienos prognozes (su 90% MC dropout intervalu):")
    for dt in savaite:
        diena = ["Pir","Ant","Tre","Ket","Pen","Ses","Sek"][dt.dayofweek]
        m_ = int(round(prog[dt]["mean"]))
        lo = int(round(prog[dt]["lo"]))
        hi = int(round(prog[dt]["hi"]))
        print(f"  {dt.date()} ({diena}): {m_:5d}   [90% CI: {lo:5d} - {hi:5d}]")

zurnalas.index.name = "Data"
zurnalas.sort_index(inplace=True)
zurnalas.to_csv(LOG_PATH)
print(f"\nZurnalas issaugotas: {LOG_PATH}")


In [ ]:
#grafika, diagnostika, metrikos, adaptacija

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

#mokymosi kreives
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist.history["loss"], label="Training loss", linewidth=2)
ax.plot(hist.history["val_loss"], label="Validation loss", linewidth=2)
ax.set_xlabel("Epocha")
ax.set_ylabel("Loss")
ax.set_title("GRU mokymosi kreives")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#gru vs baseline vs faktas
dates_2026 = zurnalas[zurnalas.index >= "2026-01-01"].index

if len(dates_2026) > 0:
    y_tmp = df_feat["Skambuciai"].copy()
    for dt in dates_2026:
        if dt not in y_tmp.index or pd.isna(y_tmp.get(dt, np.nan)):
            y_tmp.loc[dt] = zurnalas.at[dt, "Prognoze"]
    y_tmp = y_tmp.sort_index()

    atsarginis_vidurkis = df_feat["Skambuciai"].mean()
    baseline_vals = {
        "Naive (t-1)":      y_tmp.shift(1).loc[dates_2026].fillna(atsarginis_vidurkis).values,
        "Seasonal (t-7)":   y_tmp.shift(7).loc[dates_2026].fillna(atsarginis_vidurkis).values,
        "SameYear (t-364)": y_tmp.shift(364).loc[dates_2026].fillna(atsarginis_vidurkis).values,
    }
    gru_pred = zurnalas.loc[dates_2026, "Prognoze"].values.astype(float)
    ci_lo    = zurnalas.loc[dates_2026, "CI_Lower"].values.astype(float)
    ci_hi    = zurnalas.loc[dates_2026, "CI_Upper"].values.astype(float)
    faktas_vals = zurnalas.loc[dates_2026, "Faktas"].values

    fig, axes = plt.subplots(2, 1, figsize=(15, 10))

    ax = axes[0]
    colors = ["orange", "green", "red"]
    for (name, vals), color in zip(baseline_vals.items(), colors):
        ax.plot(dates_2026, vals, label=name, linewidth=1.5, alpha=0.6, color=color, linestyle="--")

    ax.fill_between(dates_2026, ci_lo, ci_hi, color="mediumpurple", alpha=0.18, label="GRU 90% MC dropout CI")
    ax.plot(dates_2026, gru_pred, label="GRU", linewidth=2.5, color="mediumpurple", marker="o", markersize=4, zorder=3)

    if not pd.isna(faktas_vals).all():
        ax.plot(dates_2026, faktas_vals, label="Faktas", linewidth=3, color="black", marker="s", markersize=5, zorder=5)

    ax.set_xlabel("Data (2026)", fontsize=11)
    ax.set_ylabel("Skambuciu kiekis", fontsize=11)
    ax.set_title(f"GRU vs baseline vs faktas (iki {dates_2026.max().date()})", fontsize=13, fontweight="bold")
    ax.legend(loc="best", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_minor_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

    ax = axes[1]
    for (name, vals), color in zip(baseline_vals.items(), colors):
        ax.plot(dates_2026, np.cumsum(vals), label=f"{name} (kumul.)", linewidth=1.5, alpha=0.6, color=color, linestyle="--")

    ax.plot(dates_2026, np.cumsum(gru_pred), label="GRU (kumul.)", linewidth=2.5, color="mediumpurple", marker="o", markersize=4, zorder=3)

    if not pd.isna(faktas_vals).all():
        faktas_cs = pd.Series(faktas_vals).cumsum()
        ax.plot(dates_2026, faktas_cs, label="Faktas (kumul.)", linewidth=3, color="black", marker="s", markersize=5, zorder=5)

    ax.set_xlabel("Data (2026)", fontsize=11)
    ax.set_ylabel("Is viso skambuciu", fontsize=11)
    ax.set_title(f"Kumuliatyvine suma (iki {dates_2026.max().date()})", fontsize=13, fontweight="bold")
    ax.legend(loc="best", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_minor_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

    #suvestine
    print("\n" + "="*70)
    print(f"SUVESTINE (iki {dates_2026.max().date()})")
    print("="*70)
    print(f"\n{'Modelis':<25} {'Vidut./d':>12}  {'Viso':>12}  {'Min':>8}  {'Max':>8}")
    print("-" * 70)
    if not pd.isna(faktas_vals).all():
        vf = faktas_vals[~pd.isna(faktas_vals)]
        print(f"{'FAKTAS':<25} {vf.mean():>12.0f}  {vf.sum():>12.0f}  {vf.min():>8.0f}  {vf.max():>8.0f}")
        print("-" * 70)
    print(f"{'GRU':<25} {gru_pred.mean():>12.0f}  {gru_pred.sum():>12.0f}  {gru_pred.min():>8.0f}  {gru_pred.max():>8.0f}")
    for name, vals in baseline_vals.items():
        print(f"{name:<25} {vals.mean():>12.0f}  {vals.sum():>12.0f}  {vals.min():>8.0f}  {vals.max():>8.0f}")
    print("="*70)
else:
    print("Zurnale dar nera 2026 prognoziu.")

#winkler score
def winkler_score(y_true, ci_lo, ci_hi, alpha=0.10):
    n = len(y_true)
    total = 0.0
    for i in range(n):
        w = ci_hi[i] - ci_lo[i]
        if y_true[i] < ci_lo[i]:
            total += w + (2.0/alpha)*(ci_lo[i]-y_true[i])
        elif y_true[i] > ci_hi[i]:
            total += w + (2.0/alpha)*(y_true[i]-ci_hi[i])
        else:
            total += w
    return total / n

uzbaigta = zurnalas.dropna(subset=["Faktas"]).copy()
if len(uzbaigta) == 0:
    print("\n2026 faktu dar nera.")
else:
    y_f = uzbaigta["Faktas"].values.astype(float)
    y_p = uzbaigta["Prognoze"].values.astype(float)
    y_hist = D_eff["Skambuciai"].values.astype(float)
    print(f"\n=== 2026 m bendros metrikos ({len(uzbaigta)} d.) ===")
    print(f"  MAE   : {mae(y_f, y_p):8.2f}")
    print(f"  RMSE  : {rmse(y_f, y_p):8.2f}")
    print(f"  MAPE  : {mape(y_f, y_p):8.2f}%")
    print(f"  sMAPE : {smape(y_f, y_p):8.2f}%")
    print(f"  MASE  : {mase(y_f, y_p, y_hist):8.2f}")

    lo = uzbaigta["CI_Lower"].values.astype(float)
    hi = uzbaigta["CI_Upper"].values.astype(float)
    picp = float(np.mean((y_f >= lo) & (y_f <= hi)))
    pinaw = float(np.mean(hi - lo) / (y_hist.max() - y_hist.min()))
    print(f"  PICP  : {picp*100:6.1f}%  (tikslas ~90%)")
    print(f"  PINAW : {pinaw:8.3f}")

    alpha_param = 0.10
    if 'CI_ALPHA' in globals():
        alpha_param = CI_ALPHA

    ws = winkler_score(y_f, lo, hi, alpha=alpha_param)
    print(f"  Winkler : {ws:8.2f}\n")

#likuciu diagnostika
try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    y_hat_train_s = modelis.predict(X_seq, verbose=0).flatten()
    y_hat_train   = inverse_scale_y(y_hat_train_s)
    y_obs_train   = inverse_scale_y(y_seq)
    resid = y_obs_train - y_hat_train

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(idx_seq, resid, color="mediumpurple", linewidth=0.8)
    axes[0].axhline(0, color="black", linewidth=0.5)
    axes[0].set_title("GRU likuciai")
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(resid, bins=50, color="mediumpurple", edgecolor="white")
    axes[1].set_title("Likuciu histograma")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    lb = acorr_ljungbox(resid, lags=[14], return_df=True)
    print(f"\nLjung-Box (lag=14): Q={lb['lb_stat'].iloc[0]:.2f}, p={lb['lb_pvalue'].iloc[0]:.4f}  "
          f"-> {'baltas triuksmas' if lb['lb_pvalue'].iloc[0] > 0.05 else 'yra autokoreliaciju'}")
except Exception as e:
    print(f"Diagnostikos nepavyko: {e}")

# adaptacija
if len(uzbaigta) >= 7:
    uzbaigta["Proc_Klaida"] = np.abs(uzbaigta["Faktas"] - uzbaigta["Prognoze"]) / uzbaigta["Faktas"] * 100
    sav_mape = uzbaigta.groupby("Savaites_Nr")["Proc_Klaida"].mean()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(sav_mape.index.astype(int), sav_mape.values, "o-", color="mediumpurple", linewidth=2, label="Savaites MAPE")
    ax.axhline(10, color="green", linestyle="--", label="10% riba")
    ax.set_xlabel("2026 m savaites nr.")
    ax.set_ylabel("MAPE (%)")
    ax.set_title("GRU adaptacija po reformos")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("\nReikia bent 1 pilnos savaites.")


In [ ]:
#dm-hln testas: gru vs baziniai

import numpy as np
from scipy import stats as sp_stats

print('=' * 70)
print('DM-HLN TESTAS')
print('=' * 70)

def dm_hln_test(e1, e2, h=1, power=2):
    e1, e2 = np.asarray(e1, float), np.asarray(e2, float)
    T = len(e1)
    d = np.abs(e1)**power - np.abs(e2)**power
    d_bar = d.mean()
    g0 = np.var(d, ddof=0)
    gs = sum(np.cov(d[k:], d[:-k], ddof=0)[0,1] for k in range(1, h))
    var_d = (g0 + 2*gs) / T
    if var_d <= 0: return np.nan, np.nan, np.nan, np.nan
    dm = d_bar / np.sqrt(var_d)
    hln = dm * np.sqrt((T+1-2*h+h*(h-1)/T)/T)
    p_dm  = 2*sp_stats.t.sf(abs(dm), df=T-1)
    p_hln = 2*sp_stats.t.sf(abs(hln), df=T-1)
    return dm, hln, p_dm, p_hln

try:
    uzb = zurnalas.dropna(subset=['Faktas'])
    if len(uzb) >= 7:
        y_fact = uzb['Faktas'].values.astype(float)
        y_pred = uzb['Prognoze'].values.astype(float)
        e_gru = y_fact - y_pred
        e_naive = y_fact - np.roll(y_fact, 1); e_naive[0] = 0
        e_snaive = y_fact - np.roll(y_fact, 7); e_snaive[:7] = 0
        e_drift = np.copy(y_fact)
        for t in range(2, len(y_fact)):
            e_drift[t] = y_fact[t-1] + (y_fact[t-1]-y_fact[0])/(t-1)
        e_drift = y_fact - e_drift
        benchmarks = {'Naive': e_naive, 'Seasonal Naive': e_snaive, 'Drift': e_drift}
        print(f'\nDuomenu tasku: {len(y_fact)}')
        print(f'{"Benchmark":<20} {"DM":>8} {"HLN":>8} {"p(DM)":>10} {"p(HLN)":>10} {"Isvada":>18}')
        print('='*70)
        results = []
        for nm, eb in benchmarks.items():
            ml = min(len(e_gru), len(eb))
            dm,hln,pd_,ph = dm_hln_test(eb[:ml], e_gru[:ml], h=7)
            results.append((nm,dm,hln,pd_,ph))
            v = 'GRU geresnis' if (not np.isnan(ph) and ph<0.05 and dm>0) else ('Benchmark' if (not np.isnan(ph) and ph<0.05 and dm<0) else 'Nesiskiria')
            print(f'{nm:<20} {dm:>8.3f} {hln:>8.3f} {pd_:>10.6f} {ph:>10.6f} {v:>18}')
        #holm-bonferroni
        print(f'\n--- Holm-Bonferroni korekcija ---')
        pv = sorted([(r[0],r[4]) for r in results if not np.isnan(r[4])], key=lambda x:x[1])
        m = len(pv)
        for rank,(nm,p) in enumerate(pv,1):
            aa = 0.05/(m-rank+1)
            print(f'  {rank}. {nm:<20} p={p:.6f}  alpha_adj={aa:.6f}  Reiksminga: {"Taip" if p<aa else "Ne"}')
    else:
        print('Per mazai duomenu DM-HLN testui.')
except Exception as e:
    print(f'DM-HLN nepavyko: {e}')


In [ ]:
#ablation + mlp kontrolinis modelis 
#tikrinam ar seku struktura is tikruju duoda nauda

import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, optimizers, callbacks

print('=' * 70)
print('Ablation studija + mlp ')
print('=' * 70)

try:
    #mlp kontrolinis modelis (be 3d seku)
    print("\n[1] Mokom MLP (be 3d seku strukturos)...")

    #plokstinam 3d -> 2d
    X_tr_flat = X_tr_s.reshape(X_tr_s.shape[0], -1)
    X_va_flat = X_va_s.reshape(X_va_s.shape[0], -1)

    mlp_model = models.Sequential([
        layers.Input(shape=(X_tr_flat.shape[1],)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='linear')
    ])
    mlp_model.compile(optimizer=optimizers.Adam(learning_rate=0.005), loss='mse')
    es_mlp = callbacks.EarlyStopping(patience=10, restore_best_weights=True)

    mlp_model.fit(X_tr_flat, y_tr_s, validation_data=(X_va_flat, y_va_s),
                  epochs=50, batch_size=32, verbose=0, callbacks=[es_mlp])

    y_va_mlp_s = mlp_model.predict(X_va_flat, verbose=0).flatten()
    mlp_val_mae = mae(inverse_scale_y(y_va_s), inverse_scale_y(y_va_mlp_s))

    y_va_gru_s = modelis.predict(X_va_s, verbose=0).flatten()
    gru_val_mae = mae(inverse_scale_y(y_va_s), inverse_scale_y(y_va_gru_s))

    print(f"Validacijos MAE:")
    print(f"   GRU MAE: {gru_val_mae:.2f}")
    print(f"   MLP MAE: {mlp_val_mae:.2f}")
    print(f"   {'GRU laimi - seku modeliavimas veikia' if gru_val_mae < mlp_val_mae else 'MLP laimi - seku struktura neduoda naudos'}")

    #ablation - kiekvienos pozymiu grupes itaka
    print("\n[2] Ablation - pozymiu grupiu itaka...")

    lag_feats = [f for f in FEATURE_COLS if 'lag' in f.lower()]
    roll_feats = [f for f in FEATURE_COLS if 'roll' in f.lower()]
    cal_feats = [f for f in FEATURE_COLS if any(k in f.lower() for k in ['day','week','month','svent','holiday', 'is_'])]

    groups = {
        "Be Lag": lag_feats,
        "Be Slankaus lango": roll_feats,
        "Be Kalendoriaus": cal_feats
    }

    print(f"{'Konfiguracija':<25} {'Val MAE':>10} {'Skirtumas':>28}")
    print("-" * 65)
    print(f"{'Visi pozymiai (Baze)':<25} {gru_val_mae:>10.2f} {'-':>28}")

    for name, feats_to_drop in groups.items():
        if not feats_to_drop: continue

        temp_features = [c for c in FEATURE_COLS if c not in feats_to_drop]
        keep_indices = [FEATURE_COLS.index(c) for c in temp_features]
        X_tr_abl = X_tr_s[:, :, keep_indices]
        X_va_abl = X_va_s[:, :, keep_indices]

        abl_model = build_gru(LOOKBACK, len(temp_features), best_params)
        es_abl = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
        abl_model.fit(X_tr_abl, y_tr_s, validation_data=(X_va_abl, y_va_s),
                      epochs=30, batch_size=int(best_params["batch_size"]), verbose=0, callbacks=[es_abl])

        y_va_abl_s = abl_model.predict(X_va_abl, verbose=0).flatten()
        abl_val_mae = mae(inverse_scale_y(y_va_s), inverse_scale_y(y_va_abl_s))
        diff = abl_val_mae - gru_val_mae

        print(f"{name:<25} {abl_val_mae:>10.2f} {f'+{diff:.2f} (svarbi grupe)' if diff > 0 else f'{diff:.2f} (grupe trukde)' :>28}")

except Exception as e:
    print(f"Ablation nepavyko: {e}")


In [ ]:
#stabilumo analize - ar klaidos stabilios laike

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

print('=' * 70)
print('Stabilumo analize')
print('=' * 70)

try:
    uzb = zurnalas.dropna(subset=['Faktas'])
    if len(uzb) >= 14:
        y_f = uzb['Faktas'].values.astype(float)
        y_p = uzb['Prognoze'].values.astype(float)
        errors = np.abs(y_f - y_p)

        #slenkanti mae (langas = 7 d)
        window = 7
        rolling_mae = pd.Series(errors).rolling(window).mean().dropna().values
        pct_errors = np.abs(y_f - y_p) / np.where(y_f != 0, y_f, 1) * 100
        rolling_mape = pd.Series(pct_errors).rolling(window).mean().dropna().values

        fig, axes = plt.subplots(1, 2, figsize=(16, 4))
        axes[0].plot(rolling_mae, color='steelblue', linewidth=1.2)
        axes[0].axhline(y=np.mean(rolling_mae), color='red', linestyle='--', label=f'Vidurkis={np.mean(rolling_mae):.1f}')
        axes[0].set_title('Slenkanti MAE (7d)')
        axes[0].set_xlabel('Periodo nr.'); axes[0].set_ylabel('MAE')
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(rolling_mape, color='darkorange', linewidth=1.2)
        axes[1].axhline(y=np.mean(rolling_mape), color='red', linestyle='--', label=f'Vidurkis={np.mean(rolling_mape):.1f}%')
        axes[1].set_title('Slenkanti MAPE (7d)')
        axes[1].set_xlabel('Periodo nr.'); axes[1].set_ylabel('MAPE %')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

        #mann-kendall trendo testas
        print('\n--- Mann-Kendall trendo testas ---')
        n = len(errors)
        s = 0
        for k in range(n-1):
            for j in range(k+1, n):
                s += np.sign(errors[j] - errors[k])
        var_s = n*(n-1)*(2*n+5)/18
        if s > 0: z_mk = (s-1)/np.sqrt(var_s)
        elif s < 0: z_mk = (s+1)/np.sqrt(var_s)
        else: z_mk = 0
        p_mk = 2*(1 - sp_stats.norm.cdf(abs(z_mk)))

        print(f'  S statistika: {s}')
        print(f'  Z statistika: {z_mk:.4f}')
        print(f'  p-reiksme: {p_mk:.6f}')
        if p_mk < 0.05:
            trend = 'didejantis' if z_mk > 0 else 'mazejantis'
            print(f'  -> reiksmingas {trend} trendas')
        else:
            print(f'  -> nera trendo - klaidos stabilios')

        #wilcoxon - pirma vs antra puse
        mid = len(errors) // 2
        if mid >= 5:
            print('\n--- Wilcoxon (1-a vs 2-a puse) ---')
            half1, half2 = errors[:mid], errors[mid:2*mid]
            w_stat, w_p = sp_stats.wilcoxon(half1, half2)
            print(f'  W statistika: {w_stat:.4f}')
            print(f'  p-reiksme: {w_p:.6f}')
            if w_p < 0.05:
                print('  -> klaidu pasiskirstymas reiksmingai skiriasi')
            else:
                print('  -> klaidu pasiskirstymas nesiskiria - modelis stabilus')
    else:
        print('Per mazai duomenu stabilumo analizei.')
except Exception as e:
    print(f'Stabilumo analize nepavyko: {e}')


In [ ]:
#priedas a: papildomas vertinimas su atsitiktiniu padalijimu

#gru veikia ant seku, todel padalijimas vyksta seku lygyje
#tai supaprastinta validacija, ne pagrindinis vertinimas

from sklearn.model_selection import train_test_split
from tensorflow.keras import callbacks as _cb_A
import pandas as pd
import numpy as np

TEST_SIZE = 0.15
RANDOM_STATE = SEED

#sekos is D_eff
X_all_A, y_all_A, idx_all_A = sukurti_sekas(D_eff, LOOKBACK)
print(f"Seku skaicius: {len(X_all_A)} (kiekviena seka: {LOOKBACK}x{len(FEATURE_COLS)})")

#atsitiktinis padalijimas
X_trA, X_teA, y_trA_s, y_teA_s, idx_trA, idx_teA = train_test_split(
    X_all_A, y_all_A, idx_all_A,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)
print(f"Train: {len(X_trA)} seku   Test: {len(X_teA)} seku")

per_year_test = pd.Series(idx_teA.year).value_counts().sort_index()
print("\nTestu skaicius pagal metus:")
for yr, cnt in per_year_test.items():
    print(f"  {yr}: {cnt} seku")

#mokymas
tf.random.set_seed(SEED)
modelis_A = build_gru(LOOKBACK, len(FEATURE_COLS), best_params)
n_val = max(1, int(0.10 * len(X_trA)))
X_trA2, y_trA2 = X_trA[:-n_val], y_trA_s[:-n_val]
X_vaA2, y_vaA2 = X_trA[-n_val:], y_trA_s[-n_val:]

es_A = _cb_A.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss")
hist_A = modelis_A.fit(
    X_trA2, y_trA2,
    validation_data=(X_vaA2, y_vaA2),
    epochs=80,
    batch_size=int(best_params["batch_size"]),
    verbose=0,
    callbacks=[es_A],
)
print(f"Priedo A GRU: {len(hist_A.history['loss'])} epochu, val_loss={min(hist_A.history['val_loss']):.5f}")

#prognozes
y_pred_te_s = modelis_A.predict(X_teA, verbose=0).flatten()
y_pred_te   = inverse_scale_y(y_pred_te_s)
y_obs_te    = inverse_scale_y(y_teA_s)

maeA  = float(np.mean(np.abs(y_obs_te - y_pred_te)))
rmseA = float(np.sqrt(np.mean((y_obs_te - y_pred_te) ** 2)))
nz    = y_obs_te != 0
mapeA = float(np.mean(np.abs((y_obs_te[nz] - y_pred_te[nz]) / y_obs_te[nz])) * 100) if nz.any() else np.nan
denom = (np.abs(y_obs_te) + np.abs(y_pred_te)) / 2.0
mk    = denom != 0
smapeA = float(np.mean(np.abs(y_obs_te[mk] - y_pred_te[mk]) / denom[mk]) * 100) if mk.any() else np.nan
ss_res = float(np.sum((y_obs_te - y_pred_te) ** 2))
ss_tot = float(np.sum((y_obs_te - y_obs_te.mean()) ** 2))
r2A    = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

print("\n=== Priedas A - rezultatai ===")
print(f"  MAE   : {maeA:8.2f}")
print(f"  RMSE  : {rmseA:8.2f}")
print(f"  MAPE  : {mapeA:8.2f}%")
print(f"  sMAPE : {smapeA:8.2f}%")
print(f"  R2    : {r2A:8.4f}")

#metrikos pagal metus
print("\nPagal metus:")
print(f"  {'Metai':<6} {'n':>5} {'MAE':>8} {'MAPE%':>8}")
for yr in sorted(set(idx_teA.year)):
    mask = (idx_teA.year == yr)
    if mask.sum() == 0: continue
    yfy = y_obs_te[mask]; ypy = y_pred_te[mask]
    ma  = float(np.mean(np.abs(yfy - ypy)))
    nz_y = yfy != 0
    mpe = float(np.mean(np.abs((yfy[nz_y] - ypy[nz_y]) / yfy[nz_y])) * 100) if nz_y.any() else np.nan
    print(f"  {yr:<6} {int(mask.sum()):>5} {ma:>8.1f} {mpe:>8.2f}")

print("\nPastaba: rezultatai optimistiniai - random padalijimas leidzia matyti kaimynines dienas per lag.")

#grafikai
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_obs_te, y_pred_te, alpha=0.4, s=15, color="mediumpurple", edgecolors="none")
mn, mx = min(y_obs_te.min(), y_pred_te.min()), max(y_obs_te.max(), y_pred_te.max())
ax.plot([mn, mx], [mn, mx], "k--", linewidth=1, label="Ideali (y=x)")
ax.set_xlabel("Faktas"); ax.set_ylabel("Prognoze")
ax.set_title(f"GRU priedas A: faktas vs prognoze (n={len(y_obs_te)})")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
sort_order = np.argsort(idx_teA)
td_sorted = idx_teA[sort_order]
yf_sorted = y_obs_te[sort_order]
yp_sorted = y_pred_te[sort_order]
step = max(1, len(td_sorted) // 80)
ax.plot(range(len(yf_sorted[::step])), yf_sorted[::step], "k-", linewidth=1.5, label="Faktas", alpha=0.8)
ax.plot(range(len(yp_sorted[::step])), yp_sorted[::step], "-", color="mediumpurple", linewidth=1.5, label="GRU", alpha=0.7)
ax.fill_between(range(len(yf_sorted[::step])), yf_sorted[::step], yp_sorted[::step], alpha=0.15, color="mediumpurple")
ax.set_xlabel("Testo dienos"); ax.set_ylabel("Skambuciai")
ax.set_title("GRU priedas A: chronologine juosta")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

#likuciu diagnostika
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

resid_app = y_obs_te - y_pred_te
n_sw = min(len(resid_app), 5000)
sw_stat, sw_p = sp_stats.shapiro(resid_app[:n_sw])

print(f'\n--- A.6 Shapiro-Wilk ---')
print(f'  W = {sw_stat:.4f},  p = {sw_p:.6f}')
if sw_p > 0.05:
    print('  -> likuciai normalus')
else:
    print('  -> likuciai nukrypsta nuo normalumo (iprasta)')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(resid_app, bins=40, density=True, color='steelblue', edgecolor='white', alpha=0.8)
xs = np.linspace(resid_app.min(), resid_app.max(), 200)
axes[0].plot(xs, sp_stats.norm.pdf(xs, resid_app.mean(), resid_app.std()), 'r-', lw=2)
axes[0].set_title('Priedas A: histograma'); axes[0].grid(True, alpha=0.3)
sp_stats.probplot(resid_app, dist='norm', plot=axes[1])
axes[1].set_title('Priedas A: Q-Q'); axes[1].grid(True, alpha=0.3)
axes[2].plot(resid_app, '-', color='steelblue', lw=0.8, alpha=0.7)
axes[2].axhline(0, color='red', ls='--', lw=1)
axes[2].axhline(resid_app.mean()+2*resid_app.std(), color='orange', ls=':', lw=1, label='2 sigma')
axes[2].axhline(resid_app.mean()-2*resid_app.std(), color='orange', ls=':', lw=1)
axes[2].set_title('Priedas A: likuciai laike'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
